In [22]:
from functions import *
import pandas as pd
import pickle
import os
from evaluate import *

In [15]:
# ticker_list = ['REE', 'SAM', 'HAP', 'GMD', 'GIL', 'TMS', 'SAV', 'DHA', 'MHC', 'HAS'] # 10 stocks with the most observations
ticker_list = ['REE', 'SAM', 'HAP'] # 3 stocks with the most observations
limits = {
    'hose':0.07,
    'hnx':0.1,
    'upcom':0.15
}

# Parameters

horizon = 10
seq_len = 1000
train_pct = 0.958

# Read and merge into 1 dataset

if "stock_data.csv" in os.listdir("data"):
    merged_df = pd.read_csv(
        os.path.join("data", "stock_data.csv"),
        index_col=None
    ).assign(
        date = lambda df : pd.to_datetime(df["date"])
    )
else:
    # Read and merge data
    hnx = pd.read_csv(os.path.join("data", "CafeF.HNX.Upto31.07.2025.csv")).assign(
        floor = "hnx"
    )
    hsx = pd.read_csv(os.path.join("data", "CafeF.HSX.Upto31.07.2025.csv")).assign(
        floor = "hose"
    )
    upcom = pd.read_csv(os.path.join("data", "CafeF.UPCOM.Upto31.07.2025.csv")).assign(
        floor = "upcom"
    )
    indexes = pd.read_csv(os.path.join("data", "CafeF.INDEX.Upto06.08.2025.csv")).assign(
        floor = "index"
    )

    # Rename columns
    hnx, hsx, upcom, indexes = [
        df.rename(columns={
            "<Ticker>":"ticker",
            "<DTYYYYMMDD>":"date",
            "<Open>":"open",
            "<High>":"high",
            "<Low>":"low",
            "<Close>":"close",
            "<Volume>":"volume"
        }) for df in [hnx, hsx, upcom, indexes]
    ]
        
    # Merge and clean data
    # UPCOM has missing tickers for some reason
    merged_df = pd.concat(
        [hnx, hsx, upcom, indexes],
        axis=0
    ).reset_index(drop=True).dropna(subset="ticker")\
    .assign(
        date=lambda df : df["date"].astype(str).apply(lambda x: datetime.strptime(x, "%Y%m%d").date())
    )
    merged_df.to_csv(
        os.path.join("data", "stock_data.csv"),
        index=False
    ) # Save merged data to save time in future runs


# Data cleaning and merging

data = merged_df.sort_values(["ticker", "date"]).assign(
    returns = lambda df : df.groupby("ticker")["close"].pct_change(),
    log_returns_pct = lambda df : np.log(df["close"] / df.groupby("ticker")["close"].shift(1))*100
)

data = data.loc[data["ticker"].str.len()==3] # Eliminate ETF, and indeces

data["limit"] = data["floor"].map(limits)
outliers = data.loc[data["returns"].abs() > data["limit"]]
clean_df = data.drop(outliers.index) # Remove outliers
print(f"% of observations removed: {round((len(outliers)/len(data))*100, 2)}%")

pivoted_data = clean_df.pivot_table(
    columns="ticker", 
    values=["open", "high", "low", "close", "returns"], 
    index="date"
)
pivoted_data.columns = pivoted_data.columns.swaplevel(0, 1)
pivoted_data = pivoted_data.sort_index(axis=1, level=0)
pivoted_data = pivoted_data.loc[:, pivoted_data.columns.get_level_values(0).isin(ticker_list)]
pivoted_data = pivoted_data.dropna() # Drop NA

data_returns = pivoted_data.loc[:, pivoted_data.columns.get_level_values(1) == "returns"]

% of observations removed: 1.05%


In [16]:
train_df, test_df = split_train_test(data_returns, train_ratio = train_pct)
realized_cov = get_rolling_realized_covariance(data_returns.values, window_size=1000)
actual_covs = realized_cov[-len(test_df):]

In [ ]:
with open(r"bekk_results\bekk_pred_covs_3A_10S.pkl", "rb") as f:
    bekk_covs = pickle.load(f)
with open(r"dcc_results\dcc_pred_covs_3A_10S.pkl", "rb") as f:
    dcc_covs = pickle.load(f)
with open(r"svr_results\svr_pred_covs_3A_10S.pkl", "rb") as f:
    svr_covs = pickle.load(f)

In [ ]:
bekk_eval = np.concatenate(np.array(bekk_covs), axis=0)[:len(actual_covs)]
dcc_eval = np.concatenate(np.array(dcc_covs), axis=0)[:len(actual_covs)]
svr_eval = np.concatenate(np.array(svr_covs), axis=0)[:len(actual_covs)]

In [25]:
bekk_frob = np.mean([frobenius_loss(true, pred) for true, pred in zip(actual_covs, bekk_eval)])
dcc_frob = np.mean([frobenius_loss(true, pred) for true, pred in zip(actual_covs, dcc_eval)])
svr_frob = np.mean([frobenius_loss(true, pred) for true, pred in zip(actual_covs, svr_eval)])